# Notebook 02 — Segment Analysis

Churn rate and key metrics broken down by Card Type and Country, including cross-tab heatmap.

In [1]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from skills.utils.plotting import save_fig, CARD_COLORS, COUNTRY_COLORS, plot_churn_bar
from skills.utils.segment_helpers import compute_churn_rate, segment_summary

sns.set_theme(style='whitegrid')


In [2]:
df = pd.read_csv('../Customer-Churn-Records.csv')
print(f"Loaded {len(df)} records")


Loaded 10000 records


## Churn Rate by Card Type

In [3]:
card_churn = compute_churn_rate(df, 'Card Type').sort_values('churn_rate', ascending=False)
fig = plot_churn_bar(card_churn, 'Card Type', title='Churn Rate by Card Type', color_map=CARD_COLORS)
save_fig(fig, '02_churn_by_card_type.png', output_dir='../outputs/figures')
plt.close(fig)
print(card_churn.to_string(index=False))


Card Type  churn_rate  n_customers
  DIAMOND    0.217790         2507
 PLATINUM    0.203607         2495
   SILVER    0.201122         2496
     GOLD    0.192646         2502


## Churn Rate by Country

In [4]:
country_churn = compute_churn_rate(df, 'Geography').sort_values('churn_rate', ascending=False)
fig = plot_churn_bar(country_churn, 'Geography', title='Churn Rate by Country', color_map=COUNTRY_COLORS)
save_fig(fig, '02_churn_by_country.png', output_dir='../outputs/figures')
plt.close(fig)
print(country_churn.to_string(index=False))


Geography  churn_rate  n_customers
  Germany    0.324432         2509
    Spain    0.166734         2477
   France    0.161747         5014


## Card Type × Country Churn Heatmap

In [5]:
pivot = df.groupby(['Card Type', 'Geography'])['Exited'].mean().unstack() * 100
fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Churn Rate (%)'})
ax.set_title('Churn Rate (%) by Card Type × Country', fontsize=13)
plt.tight_layout()
save_fig(fig, '02_churn_heatmap.png', output_dir='../outputs/figures')
plt.close(fig)


## Segment Summary — by Card Type

In [6]:
card_summary = segment_summary(df, 'Card Type').sort_values('churn_rate', ascending=False)
card_summary['churn_rate'] = card_summary['churn_rate'].round(3)
card_summary['complaint_rate'] = card_summary['complaint_rate'].round(3)
print(card_summary.to_string(index=False))
card_summary.to_csv('../outputs/tables/card_type_summary.csv', index=False)
print("\nSaved to outputs/tables/card_type_summary.csv")


Card Type  churn_rate  n_customers  avg_satisfaction  avg_balance  avg_credit_score  complaint_rate
  DIAMOND       0.218         2507          2.993618 79120.017156        651.072996           0.218
 PLATINUM       0.204         2495          3.010020 75692.707623        648.255311           0.205
   SILVER       0.201         2496          3.007212 74423.040272        650.436699           0.201
     GOLD       0.193         2502          3.044365 76695.362042        652.342526           0.193

Saved to outputs/tables/card_type_summary.csv


## Segment Summary — by Country

In [7]:
country_summary = segment_summary(df, 'Geography').sort_values('churn_rate', ascending=False)
country_summary['churn_rate'] = country_summary['churn_rate'].round(3)
country_summary['complaint_rate'] = country_summary['complaint_rate'].round(3)
print(country_summary.to_string(index=False))
country_summary.to_csv('../outputs/tables/country_summary.csv', index=False)
print("\nSaved to outputs/tables/country_summary.csv")


Geography  churn_rate  n_customers  avg_satisfaction   avg_balance  avg_credit_score  complaint_rate
  Germany       0.324         2509          3.005978 119730.116134        651.453567           0.326
    Spain       0.167         2477          3.013726  61818.147763        651.333872           0.167
   France       0.162         5014          3.017750  62092.636516        649.668329           0.162

Saved to outputs/tables/country_summary.csv


## Cross-Segment Summary — Card Type × Country

In [8]:
cross_summary = segment_summary(df, ['Card Type', 'Geography']).sort_values('churn_rate', ascending=False)
cross_summary['churn_rate'] = cross_summary['churn_rate'].round(3)
print("Top 10 highest-churn segments:")
print(cross_summary.head(10).to_string(index=False))
cross_summary.to_csv('../outputs/tables/cross_segment_summary.csv', index=False)
print("\nSaved to outputs/tables/cross_segment_summary.csv")


Top 10 highest-churn segments:
Card Type Geography  churn_rate  n_customers  avg_satisfaction   avg_balance  avg_credit_score  complaint_rate
  DIAMOND   Germany       0.340          648          3.038580 119388.489907        647.174383        0.339506
 PLATINUM   Germany       0.337          608          3.009868 119822.603668        651.580592        0.340461
   SILVER   Germany       0.313          600          2.976667 119699.010783        648.085000        0.313333
     GOLD   Germany       0.308          653          2.996937 120011.593292        658.676876        0.312404
  DIAMOND    France       0.180         1230          2.957724  66340.326715        653.810569        0.180488
   SILVER     Spain       0.177          611          2.996727  59248.649231        646.204583        0.176759
 PLATINUM     Spain       0.175          623          3.006421  64221.628796        651.629213        0.174960
  DIAMOND     Spain       0.167          629          3.017488  62625.659285     